In [1]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [4]:
data_dir = "../data/raw/plant_disease_images"


In [6]:
IMG_SIZE = 128

images = []
labels = []
class_names = []

for folder in os.listdir(data_dir):
    folder_path = os.path.join(data_dir, folder)
    
    if os.path.isdir(folder_path):
        class_names.append(folder)

for label, folder in enumerate(class_names):
    folder_path = os.path.join(data_dir, folder)
    
    for img_name in os.listdir(folder_path)[:300]:  # speed limit
        img_path = os.path.join(folder_path, img_name)
        
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        
        images.append(img)
        labels.append(label)

X = np.array(images) / 255.0
y = to_categorical(labels)

print("Total Classes:", len(class_names))
print("Dataset Shape:", X.shape)

Total Classes: 16
Dataset Shape: (4352, 128, 128, 3)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(len(class_names), activation='softmax')
])

c:\Users\Mahnoor\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 35s 323ms/step - accuracy: 0.7980 - loss: 0.6085 - val_accuracy: 0.7727 - val_loss: 0.7038
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 33s 299ms/step - accuracy: 0.8061 - loss: 0.5662 - val_accuracy: 0.7899 - val_loss: 0.6485
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 34s 312ms/step - accuracy: 0.8512 - loss: 0.4553 - val_accuracy: 0.7830 - val_loss: 0.6707
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 34s 316ms/step - accuracy: 0.8716 - loss: 0.3884 - val_accuracy: 0.7899 - val_loss: 0.7069
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 33s 303ms/step - accuracy: 0.8756 - loss: 0.3629 - val_accuracy: 0.8037 - val_loss: 0.6250
Epoch 6/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 35s 316ms/step - accuracy: 0.9135 - loss: 0.2513 - val_accuracy: 0.8266 - val_loss: 0.6269
Epoch 7/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 34s 312ms/step - accuracy: 0.9153 - loss: 0.2633 - val_accuracy: 0.7819 - val_loss: 0.6811
Epoch 8/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 33s 304ms/step - accuracy: 0.9296 - loss: 0

In [14]:
model.save("../models/disease_model.h5")

import joblib
joblib.dump(class_names, "../models/disease_classes.pkl")

print("Disease model saved!")

Disease model saved!
